In [1]:
# Now to get my income summary stats

import geopandas as gpd
from pathlib import Path
import pandas as pd
import requests
import openpyxl

In [2]:
Current_Path = Path.cwd()

Path_Root = Current_Path.parent.parent

Data_Path = Path_Root/"data"/"forecasting"
File_Path = Data_Path/"IncomeSummaryStats_1823.xlsx"

Data_Path.mkdir(parents=True, exist_ok=True)
Url = (
    "https://www.abs.gov.au/statistics/labour/"
    "earnings-and-working-conditions/personal-income-australia/"
    "2022-23/Table%201%20-%20Total%20income%2C%20earners"
    "%20and%20summary%20statistics%20by%20geography%2C%202018-"
    "19%20to%202022-23.xlsx"
)

Response = requests.get(
    Url,
    timeout=60,
)

Response.raise_for_status()

with open(File_Path, "wb") as File:
    File.write(Response.content)


print(f"Downloaded successfully to: {File_Path}")

Downloaded successfully to: /Users/ben/Documents/GitHub/real-estate-industry-project/data/forecasting/IncomeSummaryStats_1823.xlsx


In [3]:
# Now to convert to a CSV file

Uncleaned_Data = pd.read_excel(File_Path,
                               sheet_name="Table 1.4",
                               header=None
                               )
Csv1_File_Path = Data_Path/"IncomeSummaryStats_1823_Uncleaned.csv"

Uncleaned_Data.to_csv(Csv1_File_Path)

In [4]:
Uncleaned_Data = pd.read_csv(Csv1_File_Path)
Uncleaned_Data.head(10)

,Unnamed: 0,0,1,2,3,4,5,6,7,8,...,17,18,19,20,21,22,23,24,25,26
0,0,Australian Bureau of Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Personal Income in Australia: Table 1. Total i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Released at 11.30am (Canberra time) 14 Novembe...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Table 1.4,"Total income - Number of earners, median age o...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,NaN,NaN,Earners (persons),NaN,NaN,NaN,NaN,Median age of earners (years),NaN,...,Median ($),NaN,NaN,NaN,NaN,Mean ($),NaN,NaN,NaN,NaN
6,6,SA2,SA2 NAME,2018-19,2019-20,2020-21,2021-22,2022-23,2018-19,2019-20,...,2018-19,2019-20,2020-21,2021-22,2022-23,2018-19,2019-20,2020-21,2021-22,2022-23
7,7,Australia,NaN,15744685,15913540,16005361,16535703,17278093,41,41,...,49096,50103,52925,55062,58216,63448,64858,68406,72187,74995
8,8,New South Wales,NaN,5000145,5035692,5018669,5142655,5362189,41,41,...,49522,50567,53905,55105,58909,66310,67677,71919,75511,78390
9,9,101021007,Braidwood,2586,2678,2710,2780,2817,50,50,...,39484,42878,45806,48324,49963,49987,60017,67472,61492,63025


In [5]:
######################################################################
# Now to clean this data, renaming coloumns, and filtering victorian
# Sa2 codes
######################################################################

# Row 6 contains the financial-year labels.
# Actual observations begin at row 7.
Income_Data = Uncleaned_Data.iloc[7:].copy()

# Remove the unnecessary first column.
Income_Data = Income_Data.drop(
    columns=["Unnamed: 0"]
)

# Remove ABS Notes / Empty Rows

# Valid SA2 observations have a 9-digit numeric SA2 code.
Income_Data = Income_Data[
    Income_Data["0"]
    .astype(str)
    .str.fullmatch(r"\d{9}")
].copy()

# Create Column Names

Years = [
    "2018_19",
    "2019_20",
    "2020_21",
    "2021_22",
    "2022_23",
]

Column_Names = [
    "SA2_Code",
    "SA2_Name",
]

Column_Names += [
    f"Earners_{Year}"
    for Year in Years
]

Column_Names += [
    f"Median_Age_{Year}"
    for Year in Years
]

Column_Names += [
    f"Total_Income_{Year}"
    for Year in Years
]

Column_Names += [
    f"Median_Income_{Year}"
    for Year in Years
]

Column_Names += [
    f"Mean_Income_{Year}"
    for Year in Years
]

Income_Data.columns = Column_Names

# Clean Data Types


# Keep SA2 codes as strings because they are
# geographic identifiers, not quantities.
Income_Data["SA2_Code"] = (
    Income_Data["SA2_Code"]
    .astype(str)
)


# Convert all income/statistic columns to numeric.
Numeric_Columns = Income_Data.columns[
    2:
]

Income_Data[Numeric_Columns] = (
    Income_Data[Numeric_Columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

Income_Data = Income_Data.reset_index(
    drop=True
)


In [6]:
###########################################################
# Now to filter out victorian sa2 codes, median age,  
# median income and mean income
###########################################################

Income_Data = Income_Data[
    Income_Data["SA2_Code"].str.startswith("2")
].copy()

Income_Data = Income_Data.reset_index(
    drop=True
)

Income_Data = Income_Data[
    [
        Column
        for Column in Income_Data.columns
        if (
            Column in ["SA2_Code", "SA2_Name"]
            or "Median_Age" in Column
            or "Median_Income" in Column
            or "Mean_Income" in Column
        )
    ]
].copy()

Income_Data

,SA2_Code,SA2_Name,Median_Age_2018_19,Median_Age_2019_20,Median_Age_2020_21,Median_Age_2021_22,Median_Age_2022_23,Median_Income_2018_19,Median_Income_2019_20,Median_Income_2020_21,Median_Income_2021_22,Median_Income_2022_23,Mean_Income_2018_19,Mean_Income_2019_20,Mean_Income_2020_21,Mean_Income_2021_22,Mean_Income_2022_23
0,201011001,Alfredton,41.0,41.0,40.0,40.0,40.0,51620.0,52777.0,55499.0,58316.0,60841.0,61305.0,63327.0,66618.0,69685.0,72489.0
1,201011002,Ballarat,46.0,47.0,46.0,46.0,47.0,51187.0,50968.0,53232.0,55618.0,58553.0,74463.0,75117.0,80268.0,82715.0,86356.0
2,201011005,Buninyong,45.0,45.0,46.0,46.0,46.0,49846.0,51572.0,53959.0,55278.0,57501.0,60979.0,62365.0,65468.0,68664.0,70993.0
3,201011006,Delacombe,38.0,38.0,37.0,36.0,35.0,47940.0,49810.0,51041.0,53390.0,56390.0,52505.0,54270.0,56825.0,59595.0,62175.0
4,201011007,Smythes Creek,45.0,45.0,45.0,45.0,46.0,50938.0,52364.0,55547.0,55423.0,55311.0,57807.0,59026.0,62850.0,64520.0,66903.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517,217031476,Otway,49.0,50.0,51.0,50.0,50.0,32420.0,33597.0,38795.0,40989.0,41764.0,43608.0,45632.0,52824.0,54543.0,56136.0
518,217041477,Moyne - East,45.0,45.0,45.0,45.0,44.0,43158.0,43759.0,47460.0,49857.0,50118.0,52694.0,55703.0,56891.0,61049.0,64344.0
519,217041478,Moyne - West,47.0,47.0,48.0,48.0,48.0,43243.0,44966.0,48614.0,51579.0,51417.0,53248.0,55745.0,59513.0,64874.0,63858.0
520,217041479,Warrnambool - North,42.0,42.0,42.0,42.0,42.0,45632.0,47821.0,50159.0,51796.0,55031.0,52602.0,55573.0,58183.0,60799.0,63722.0


In [7]:
# Now to write this to a file

Csv2_File_Path = Data_Path/"IncomeSummaryStats_1823_Cleaned.csv"
Income_Data.to_csv(Csv2_File_Path)

In [ ]:
# Now since i have found more income data, imma combine this to give me 
# more income based data :)


Url3 = ("https://www.abs.gov.au/statistics/labour/"
        "earnings-and-working-conditions/personal-"
        "income-australia/2011-12-2016-17/"
        "6124055002ds0001_2019.xls"
)

Url4 = ("https://www.abs.gov.au/statistics/labour/"
        "earnings-and-working-conditions/personal-"
        "income-australia/2014-15-2018-19/"
        "6524055002_DO001.xlsx"
)


File_Path3 = Data_Path/"IncomeSummaryStats_1117.xls"

File_Path4 = Data_Path/"IncomeSummaryStats_1419.xlsx"


Responce3 = requests.get(
    Url3,
    timeout=60
)

Responce4 = requests.get(
    Url4,
    timeout=60
)


with open(File_Path3, "wb") as File:
    File.write(Responce3.content)

with open(File_Path4, "wb") as File:
    File.write(Responce4.content)

print(f"Downloaded successfully to: {File_Path3}")
print(f"Downloaded successfully to: {File_Path4}")

Downloaded successfully to: /Users/ben/Documents/GitHub/real-estate-industry-project/data/forecasting/IncomeSummaryStats_0511.xls
Downloaded successfully to: /Users/ben/Documents/GitHub/real-estate-industry-project/data/forecasting/IncomeSummaryStats_1117.xls
Downloaded successfully to: /Users/ben/Documents/GitHub/real-estate-industry-project/data/forecasting/IncomeSummaryStats_1419.xlsx


In [ ]:
Unclean_Data1117 = pd.read_excel(File_Path3,
                                 sheet_name="Table 1.4",
                                 header=None
)

Unclean_Data1419 = pd.read_excel(File_Path4,
                                 sheet_name="Table 1.4",
                                 header=None)



In [ ]:
Csv_1117_FilePath = Data_Path/"IncomeSummaryStats_1117_Unclean.csv"
Csv_1419_FilePath = Data_Path/"IncomeSummaryStats_1419_Unclean.csv"

Unclean_Data1117.to_csv(Csv_1117_FilePath)
Unclean_Data1419.to_csv(Csv_1419_FilePath)

In [17]:
####### Now to take these dataframes, get the featuers i want
####### And make one big dataframe lasting more than 10 years of 
####### Economic data

Income_Data_1117 = pd.read_csv(Csv_1117_FilePath)
Income_Data_1419 = pd.read_csv(Csv_1419_FilePath)

In [ ]:
# Now to get the 2011-12 to 2016-17 data into the form i want it in such that
# It matches the 2018-23 income data

Income_Data_1117 = Income_Data_1117[
    [
        # SA2 information
        "0",
        "1",

        # Median Age
        "8",
        "9",
        "10",
        "11",
        "12",
        "13",

        # Median Income
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",

        # Mean Income
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
    ]
].copy()


Years = [
    "2011_12",
    "2012_13",
    "2013_14",
    "2014_15",
    "2015_16",
    "2016_17",
]

Column_Names = [
    "SA2_Code",
    "SA2_Name",
]

Column_Names += [
    f"Median_Age_{Year}"
    for Year in Years
]

Column_Names += [
    f"Median_Income_{Year}"
    for Year in Years
]

Column_Names += [
    f"Mean_Income_{Year}"
    for Year in Years
]

Income_Data_1117.columns = Column_Names

Income_Data_1117 = Income_Data_1117[
    Income_Data_1117["SA2_Code"]
    .astype(str)
    .str.fullmatch(r"\d{9}")
].copy()

Income_Data_1117["SA2_Code"] = (
    Income_Data_1117["SA2_Code"]
    .astype(str)
)

Income_Data_1117 = Income_Data_1117[
    Income_Data_1117["SA2_Code"]
    .str.startswith("2")
].copy()

Numeric_Columns = (
    Income_Data_1117.columns[2:]
)

Income_Data_1117[Numeric_Columns] = (
    Income_Data_1117[Numeric_Columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

Income_Data_1117 = (
    Income_Data_1117
    .reset_index(drop=True)
)

In [22]:
# Now to extact the 17-18 economic data :)

Income_Data_1718 = Income_Data_1419[
    [
        "0",   # SA2 Code
        "1",   # SA2 Name
        "10",  # Median Age 2017-18
        "20",  # Median Income 2017-18
        "25",  # Mean Income 2017-18
    ]
].copy()

Income_Data_1718.columns = [
    "SA2_Code",
    "SA2_Name",
    "Median_Age_2017_18",
    "Median_Income_2017_18",
    "Mean_Income_2017_18",
]

Income_Data_1718 = Income_Data_1718[
    Income_Data_1718["SA2_Code"]
    .astype(str)
    .str.fullmatch(r"\d{9}")
].copy()

Income_Data_1718["SA2_Code"] = (
    Income_Data_1718["SA2_Code"]
    .astype(str)
)

Income_Data_1718 = Income_Data_1718[
    Income_Data_1718["SA2_Code"]
    .str.startswith("2")
].copy()

Numeric_Columns = [
    "Median_Age_2017_18",
    "Median_Income_2017_18",
    "Mean_Income_2017_18",
]

Income_Data_1718[Numeric_Columns] = (
    Income_Data_1718[Numeric_Columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

Income_Data_1718 = (
    Income_Data_1718
    .reset_index(drop=True)
)



In [23]:
Income_Data_1823 = pd.read_csv(Csv2_File_Path)

In [25]:
# Just a step to make sure merging is all good :)

Income_Data_1117["SA2_Code"] = (
    Income_Data_1117["SA2_Code"]
    .astype(str)
)

Income_Data_1718["SA2_Code"] = (
    Income_Data_1718["SA2_Code"]
    .astype(str)
)

Income_Data_1823["SA2_Code"] = (
    Income_Data_1823["SA2_Code"]
    .astype(str)
)

In [26]:
# Now to merge and reorder my data :)

Income_Data = Income_Data_1117.merge(
    Income_Data_1718.drop(
        columns=["SA2_Name"],
        errors="ignore",
    ),
    on="SA2_Code",
    how="outer",
)

Income_Data = Income_Data.merge(
    Income_Data_1823.drop(
        columns=["SA2_Name"],
        errors="ignore",
    ),
    on="SA2_Code",
    how="outer",
)

In [28]:
# Now for the reordering of rows and that :)

Median_Age_Columns = sorted(
    [
        Column
        for Column in Income_Data.columns
        if Column.startswith("Median_Age_")
    ]
)

Median_Income_Columns = sorted(
    [
        Column
        for Column in Income_Data.columns
        if Column.startswith("Median_Income_")
    ]
)

Mean_Income_Columns = sorted(
    [
        Column
        for Column in Income_Data.columns
        if Column.startswith("Mean_Income_")
    ]
)

Income_Data = Income_Data[
    [
        "SA2_Code",
        "SA2_Name",
    ]
    + Median_Age_Columns
    + Median_Income_Columns
    + Mean_Income_Columns
].copy()

In [30]:
Final_File_Path = Data_Path/"IncomeSummaryStats_1123.csv"

Income_Data.to_csv(Final_File_Path)